In [ ]:
import os, json
import torch
import transformers
import pandas as pd
import numpy as np

import transformers 
from transformers import AutoTokenizer

import MeMoHF
from MeMoHF.modelling_memo_tokenizer import MeMoTokenizer
from MeMoHF.modelling_memo_configuration import MeMoConfig
from MeMoHF.modelling_memo import MeMoForCausalLM
from MeMoHF.evaluating_memo import Evaluation
from MeMoHF.utils import (
    seed_everything,
    load_model_and_tokenizer,
    save_data_to_disk,
    load_from_disk
)

import datasets 
from datasets import Dataset, DatasetDict, Features, Value, load_dataset, load_from_disk, concatenate_datasets



/opt/ubuntu-cuda/anaconda3/envs/EasyEdit/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/opt/ubuntu-cuda/anaconda3/envs/EasyEdit/lib/python3.11/site-packages/transformers/utils/hub.py:111: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [2]:
from learning_evaluation import create_large_sample

In [ ]:
data_path = '~/hf_cache/datasets/wikipedia/20200501.en/1.0.0/009f923d9b6dd00c00c8cdc7f408f2b47f45dd4f5fb7982a21f9448f4afbe475/wikipedia-train.arrow'
data = dict(
    train=Dataset.from_file(data_path)
)

In [4]:
data['train'].select_columns('text')

Dataset({
    features: ['text'],
    num_rows: 6078422
})

In [5]:
import learning_evaluation
from learning_evaluation import create_all_datasets

create_all_datasets(main_data_dir='training_data')


saving data to training_data/wiki_large
data:
Dataset({
    features: ['text'],
    num_rows: 300000
})
creating training_data/wiki_large...


Saving the dataset (2/2 shards): 100%|██████████| 300000/300000 [00:08<00:00, 36765.31 examples/s]


saving data to training_data/samples/n=1000
data:
Dataset({
    features: ['text'],
    num_rows: 1000
})
creating training_data/samples/n=1000...


Saving the dataset (1/1 shards): 100%|██████████| 1000/1000 [00:00<00:00, 91347.33 examples/s] 


saving data to training_data/samples/n=3000
data:
Dataset({
    features: ['text'],
    num_rows: 3000
})
creating training_data/samples/n=3000...


...

Saving the dataset (1/1 shards): 100%|██████████| 10000/10000 [00:00<00:00, 104954.95 examples/s]


saving data to training_data/samples/n=20000
data:
Dataset({
    features: ['text'],
    num_rows: 20000
})
creating training_data/samples/n=20000...


Saving the dataset (1/1 shards): 100%|██████████| 20000/20000 [00:00<00:00, 108562.85 examples/s]


saving data to training_data/samples/n=30000
data:
Dataset({
    features: ['text'],
    num_rows: 30000
})
creating training_data/samples/n=30000...


Saving the dataset (1/1 shards): 100%|██████████| 30000/30000 [00:00<00:00, 202198.15 examples/s]


In [ ]:
def convert_text_into_cfg(text):
    cfg_list = [
        (param.split('=['))
        for param in text.split(']-')
    ]
    return {
        param[0]:param[1]
        for param in cfg_list
    }

convert_text_into_cfg(os.path.basename('bla/di/bla/max_length=[1024]-d=[1024]-l=[4]-h=[4]-batch_size=[64]-save_every_k_batches=[3]-data_name=[n=1000]-seed=[42]-batch_id=[3]'))

'max_length=[1024]-d=[1024]-l=[4]-h=[4]-batch_size=[64]-save_every_k_batches=[3]-data_name=[n=1000]-seed=[42]-batch_id=[3]'

In [ ]:
import pandas as pd

# Sample DataFrame
df = pd.DataFrame([
    {'lr': 0.01, 'batch_size': 32, 'optimizer': 'adam'},
    {'lr': 0.001, 'batch_size': 64, 'optimizer': 'sgd'},
    {'lr': 0.01, 'batch_size': 64, 'optimizer': 'adam'},
])

# Dictionary with a subset of keys
partial_experiment = {'lr': 0.01, 'optimizer': 'adam'}

# Subset DataFrame to the relevant columns and compare
match = (df[partial_experiment.keys()] == pd.Series(partial_experiment)).all(axis=1)

# Check if any row matches the subset
exists = match.any()

print("Subset match exists:", exists)

Subset match exists: False


In [11]:
import pandas as pd

# Existing DataFrame
df = pd.DataFrame([
    {'lr': 0.01, 'batch_size': 32, 'optimizer': 'adam'},
    {'lr': 0.001, 'batch_size': 64, 'optimizer': 'sgd'},
])

# New row as a dictionary
new_row = {'lr': 0.005, 'batch_size': 128, 'optimizer': 'adam'}

# Add the new row
df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)

print(df)

      lr  batch_size optimizer
0  0.010          32      adam
1  0.001          64       sgd
2  0.005         128      adam
